# Reshaping

En este capítulo aprenderás a transformar y reorganizar tus datos para facilitar el análisis y la visualización. Exploraremos cómo cambiar la forma y dimensiones de los DataFrames usando métodos de Polars como `pivot`, `unpivot`, `transpose`, `explode` y `partition_by`. Estas técnicas te permitirán adaptar tus datos a diferentes necesidades analíticas y mejorar el rendimiento computacional. Además, se indicará cómo obtener los archivos de ejemplo necesarios para practicar los conceptos presentados.

## Pivot to a Wider DataFrame

## Argumentos de `df.pivot()`

| Argumento           | Descripción                                                                                   |
|---------------------|----------------------------------------------------------------------------------------------|
| `index`             | Columna(s) que se usarán como índice en el DataFrame resultante.                             |
| `columns`           | Columna(s) cuyos valores únicos se convertirán en nuevas columnas.                           |
| `values`            | Columna(s) cuyos valores se colocarán en las celdas de la tabla pivote.                      |
| `aggregate_function`| Función para agregar valores duplicados (por ejemplo, `sum`, `mean`, `first`, etc.).         |
| `maintain_order`    | Si es `True`, mantiene el orden original de las columnas y filas.                            |
| `sort_columns`      | Si es `True`, ordena alfabéticamente las columnas resultantes.                               |
| `separator`         | Cadena utilizada para unir nombres de columnas múltiples en el resultado.                    |


In [39]:
import polars as pl

In [40]:
grades_wide = pl.DataFrame(
    {
        "student": ["Jeroen", "Thijs", "Ritchie"],
        "math": [85, 78, 92],
        "science": [90, 82, 85],
        "history": [88, 80, 87],
    }
)

grades_wide

student,math,science,history
str,i64,i64,i64
"""Jeroen""",85,90,88
"""Thijs""",78,82,80
"""Ritchie""",92,85,87


In [41]:
grades_long = pl.DataFrame(
    {
        "student": [
            "Jeroen",
            "Jeroen",
            "Jeroen",
            "Thijs",
            "Thijs",
            "Thijs",
            "Ritchie",
            "Ritchie",
            "Ritchie",
        ],
        "subject": [
            "Math",
            "Science",
            "History",
            "Math",
            "Science",
            "History",
            "Math",
            "Science",
            "History",
        ],
        "grade": [85, 90, 88, 78, 82, 80, 92, 85, 87],
    }
)

grades_long

student,subject,grade
str,str,i64
"""Jeroen""","""Math""",85
"""Jeroen""","""Science""",90
"""Jeroen""","""History""",88
"""Thijs""","""Math""",78
"""Thijs""","""Science""",82
"""Thijs""","""History""",80
"""Ritchie""","""Math""",92
"""Ritchie""","""Science""",85
"""Ritchie""","""History""",87


In [42]:
grades_long.pivot(index="student", on="subject", values="grade")

student,Math,Science,History
str,i64,i64,i64
"""Jeroen""",85,90,88
"""Thijs""",78,82,80
"""Ritchie""",92,85,87


In [43]:
multiple_grades = pl.DataFrame(
    {
        "student": [
            "Jeroen",
            "Jeroen",
            "Jeroen",
            "Jeroen",
            "Jeroen",
            "Jeroen",
            "Thijs",
            "Thijs",
            "Thijs",
            "Thijs",
            "Thijs",
            "Thijs",
        ],
        "subject": [
            "Math",
            "Math",
            "Math",
            "Science",
            "Science",
            "Science",
            "Math",
            "Math",
            "Math",
            "Science",
            "Science",
            "Science",
        ],
        "grade": [85, 88, 85, 60, 66, 63, 51, 79, 62, 82, 85, 82],
    }
)
with pl.Config(tbl_rows=-1):
    print(multiple_grades)

shape: (12, 3)
┌─────────┬─────────┬───────┐
│ student ┆ subject ┆ grade │
│ ---     ┆ ---     ┆ ---   │
│ str     ┆ str     ┆ i64   │
╞═════════╪═════════╪═══════╡
│ Jeroen  ┆ Math    ┆ 85    │
│ Jeroen  ┆ Math    ┆ 88    │
│ Jeroen  ┆ Math    ┆ 85    │
│ Jeroen  ┆ Science ┆ 60    │
│ Jeroen  ┆ Science ┆ 66    │
│ Jeroen  ┆ Science ┆ 63    │
│ Thijs   ┆ Math    ┆ 51    │
│ Thijs   ┆ Math    ┆ 79    │
│ Thijs   ┆ Math    ┆ 62    │
│ Thijs   ┆ Science ┆ 82    │
│ Thijs   ┆ Science ┆ 85    │
│ Thijs   ┆ Science ┆ 82    │
└─────────┴─────────┴───────┘


In [44]:
multiple_grades.pivot(
    index="student",
    on="subject",
    values="grade",
    aggregate_function="mean",
)

student,Math,Science
str,f64,f64
"""Jeroen""",86.0,63.0
"""Thijs""",64.0,83.0


In [45]:
multiple_grades.pivot(
    index="student",
    on="subject",
    values="grade",
    aggregate_function=pl.element().max() - pl.element().min(),
)

student,Math,Science
str,i64,i64
"""Jeroen""",3,6
"""Thijs""",28,3


In [46]:
lf = pl.LazyFrame(
    {
        "col1": ["a", "a", "a", "b", "b", "b"],
        "col2": ["x", "x", "x", "x", "y", "y"],
        "col3": [6, 7, 3, 2, 5, 7],
    }
)

index = pl.col("col1")
on = pl.col("col2")
values = pl.col("col3")
unique_column_values = ["x", "y"]
aggregate_function = lambda col: col.tanh().mean()

lf.group_by(index).agg(
    aggregate_function(values.filter(on == value)).alias(value)
    for value in unique_column_values
).collect()

col1,x,y
str,f64,f64
"""a""",0.998347,null
"""b""",0.964028,0.999954


## Unpivot to a Longer DataFrame

| Argumento        | Descripción                                                                                  |
|------------------|---------------------------------------------------------------------------------------------|
| `id_vars`        | Columnas que se mantendrán fijas (identificadores), no se transforman.                      |
| `value_vars`     | Columnas que se convertirán en filas largas (variables que se "deshacen"/apilan).           |
| `variable_name`  | Nombre de la nueva columna que contendrá los nombres de las columnas originales apiladas.    |
| `value_name`     | Nombre de la nueva columna que contendrá los valores de las columnas apiladas.              |

In [47]:
grades_wide

student,math,science,history
str,i64,i64,i64
"""Jeroen""",85,90,88
"""Thijs""",78,82,80
"""Ritchie""",92,85,87


In [48]:
grades_wide.unpivot(
    index=["student"],
    on=["math", "science", "history"],
    variable_name="subject",
    value_name="grade",
)

student,subject,grade
str,str,i64
"""Jeroen""","""math""",85
"""Thijs""","""math""",78
"""Ritchie""","""math""",92
"""Jeroen""","""science""",90
"""Thijs""","""science""",82
"""Ritchie""","""science""",85
"""Jeroen""","""history""",88
"""Thijs""","""history""",80
"""Ritchie""","""history""",87


In [49]:
df = pl.DataFrame(
    {
        "student": ["Jeroen", "Thijs", "Ritchie", "Jeroen", "Thijs", "Ritchie"],
        "class": [
            "Math101",
            "Math101",
            "Math101",
            "Math102",
            "Math102",
            "Math102",
        ],
        "age": [20, 21, 22, 20, 21, 22],
        "semester": ["Fall", "Fall", "Fall", "Spring", "Spring", "Spring"],
        "math": [85, 78, 92, 88, 79, 95],
        "science": [90, 82, 85, 92, 81, 87],
        "history": [88, 80, 87, 85, 82, 89],
    }
)
df

student,class,age,semester,math,science,history
str,str,i64,str,i64,i64,i64
"""Jeroen""","""Math101""",20,"""Fall""",85,90,88
"""Thijs""","""Math101""",21,"""Fall""",78,82,80
"""Ritchie""","""Math101""",22,"""Fall""",92,85,87
"""Jeroen""","""Math102""",20,"""Spring""",88,92,85
"""Thijs""","""Math102""",21,"""Spring""",79,81,82
"""Ritchie""","""Math102""",22,"""Spring""",95,87,89


In [50]:
df.unpivot(
    index=["student", "class", "age", "semester"],
    on=["math", "science", "history"],
    variable_name="subject",
    value_name="grade",
)

student,class,age,semester,subject,grade
str,str,i64,str,str,i64
"""Jeroen""","""Math101""",20,"""Fall""","""math""",85
"""Thijs""","""Math101""",21,"""Fall""","""math""",78
"""Ritchie""","""Math101""",22,"""Fall""","""math""",92
"""Jeroen""","""Math102""",20,"""Spring""","""math""",88
"""Thijs""","""Math102""",21,"""Spring""","""math""",79
…,…,…,…,…,…
"""Thijs""","""Math101""",21,"""Fall""","""history""",80
"""Ritchie""","""Math101""",22,"""Fall""","""history""",87
"""Jeroen""","""Math102""",20,"""Spring""","""history""",85


## Transposing

.transpose() invierte filas y columnas en un DataFrame, convirtiendo las filas en columnas y viceversa. Esto es útil para reorganizar datos y facilitar ciertos tipos de análisis.

| Argumento      | Descripción                                                                                  |
|----------------|---------------------------------------------------------------------------------------------|
| `include_header` | Si es `True`, incluye los nombres de las columnas como la primera fila del DataFrame transpuesto. |
| `column_names`   | Lista de nombres para las columnas del DataFrame transpuesto. Si no se proporciona, se usarán índices numéricos. |
|`header_name`    | Nombre para la columna que contendrá los nombres de las columnas originales si `include_header` es `True`. |

In [51]:
grades_wide

student,math,science,history
str,i64,i64,i64
"""Jeroen""",85,90,88
"""Thijs""",78,82,80
"""Ritchie""",92,85,87


In [52]:
report_columns = (f'report_{i+1}' for i, _ in enumerate(df.columns))
grades_wide.transpose(
    include_header=True,
    header_name="original_header",
    column_names=report_columns,
)

original_header,report_1,report_2,report_3
str,str,str,str
"""student""","""Jeroen""","""Thijs""","""Ritchie"""
"""math""","""85""","""78""","""92"""
"""science""","""90""","""82""","""85"""
"""history""","""88""","""80""","""87"""


## Exploding

In [53]:
grades_nested = pl.DataFrame(
    {
        "student": ["Jeroen", "Thijs", "Ritchie"],
        "math": [[85, 90, 88], [78, 82, 80], [92, 85, 87]],
    }
)

grades_nested

student,math
str,list[i64]
"""Jeroen""","[85, 90, 88]"
"""Thijs""","[78, 82, 80]"
"""Ritchie""","[92, 85, 87]"


In [54]:
grades_nested.explode("math")

student,math
str,i64
"""Jeroen""",85
"""Jeroen""",90
"""Jeroen""",88
"""Thijs""",78
"""Thijs""",82
"""Thijs""",80
"""Ritchie""",92
"""Ritchie""",85
"""Ritchie""",87


In [55]:
grades_nested = pl.DataFrame(
    {
        "student": ["Jeroen", "Thijs", "Ritchie"],
        "math": [[85, 90, 88], [78, 82, 80], [92, 85, 87]],
        "science": [[85, 90, 88], [78, 82], [92, 85, 87]],
        "history": [[85, 90, 88], [78, 82], [92, 85, 87]],
    }
)

grades_nested

student,math,science,history
str,list[i64],list[i64],list[i64]
"""Jeroen""","[85, 90, 88]","[85, 90, 88]","[85, 90, 88]"
"""Thijs""","[78, 82, 80]","[78, 82]","[78, 82]"
"""Ritchie""","[92, 85, 87]","[92, 85, 87]","[92, 85, 87]"


In [56]:
#grades_nested.explode('math', 'science', 'history') ShapeError: exploded columns must have matching element counts

In [65]:
grades_nested_long = grades_nested.unpivot(
    on=['math', 'science', 'history'],
    index='student',
    variable_name='subject',
    value_name='grade'
)
grades_nested_long

student,subject,grade
str,str,list[i64]
"""Jeroen""","""math""","[85, 90, 88]"
"""Thijs""","""math""","[78, 82, 80]"
"""Ritchie""","""math""","[92, 85, 87]"
"""Jeroen""","""science""","[85, 90, 88]"
"""Thijs""","""science""","[78, 82]"
"""Ritchie""","""science""","[92, 85, 87]"
"""Jeroen""","""history""","[85, 90, 88]"
"""Thijs""","""history""","[78, 82]"
"""Ritchie""","""history""","[92, 85, 87]"


In [66]:
grades_nested_long.explode('grade')

student,subject,grade
str,str,i64
"""Jeroen""","""math""",85
"""Jeroen""","""math""",90
"""Jeroen""","""math""",88
"""Thijs""","""math""",78
"""Thijs""","""math""",82
…,…,…
"""Thijs""","""history""",78
"""Thijs""","""history""",82
"""Ritchie""","""history""",92


In [69]:
nested_lists = pl.DataFrame(
    {
        "id": [1, 2],
        "nested_value": [[["a", "b"]], [["c"], ["d", "e"]]],
    },
    strict=False,
)
nested_lists

id,nested_value
i64,list[list[str]]
1,"[[""a"", ""b""]]"
2,"[[""c""], [""d"", ""e""]]"


In [72]:
nested_lists.explode('nested_value')

id,nested_value
i64,list[str]
1,"[""a"", ""b""]"
2,"[""c""]"
2,"[""d"", ""e""]"


In [73]:
nested_lists.explode('nested_value').explode('nested_value')

id,nested_value
i64,str
1,"""a"""
1,"""b"""
2,"""c"""
2,"""d"""
2,"""e"""


## Partition into Multiple DataFrames

| Argumento      | Descripción                                                                                                   |
|----------------|--------------------------------------------------------------------------------------------------------------|
| `by`           | Columna(s) por las que se particionará el DataFrame. Puede ser una cadena, lista de cadenas o expresiones.   |
| `*more_by`     | Columnas adicionales para particionar, especificadas como argumentos separados.                              |
| `maintain_order` | Si es `True`, mantiene el orden original de las filas dentro de cada partición. Por defecto es `False`.     |
| `include_key`  | Si es `True`, cada partición será una tupla `(clave, DataFrame)`. Si es `False`, solo se devuelve el DataFrame.|
| `as_dict`      | Si es `True`, devuelve un diccionario `{clave: DataFrame}` en vez de una lista de DataFrames o tuplas.        |

In [75]:
sales = pl.DataFrame(
    {
        "OrderID": [1, 2, 3, 4, 5, 6],
        "Product": ["A", "B", "A", "C", "B", "A"],
        "Quantity": [10, 5, 8, 7, 3, 12],
        "Region": ["North", "South", "North", "West", "South", "West"],
    }
)
sales

OrderID,Product,Quantity,Region
i64,str,i64,str
1,"""A""",10,"""North"""
2,"""B""",5,"""South"""
3,"""A""",8,"""North"""
4,"""C""",7,"""West"""
5,"""B""",3,"""South"""
6,"""A""",12,"""West"""


In [76]:
sales.partition_by('Region')

[shape: (2, 4)
 ┌─────────┬─────────┬──────────┬────────┐
 │ OrderID ┆ Product ┆ Quantity ┆ Region │
 │ ---     ┆ ---     ┆ ---      ┆ ---    │
 │ i64     ┆ str     ┆ i64      ┆ str    │
 ╞═════════╪═════════╪══════════╪════════╡
 │ 1       ┆ A       ┆ 10       ┆ North  │
 │ 3       ┆ A       ┆ 8        ┆ North  │
 └─────────┴─────────┴──────────┴────────┘,
 shape: (2, 4)
 ┌─────────┬─────────┬──────────┬────────┐
 │ OrderID ┆ Product ┆ Quantity ┆ Region │
 │ ---     ┆ ---     ┆ ---      ┆ ---    │
 │ i64     ┆ str     ┆ i64      ┆ str    │
 ╞═════════╪═════════╪══════════╪════════╡
 │ 2       ┆ B       ┆ 5        ┆ South  │
 │ 5       ┆ B       ┆ 3        ┆ South  │
 └─────────┴─────────┴──────────┴────────┘,
 shape: (2, 4)
 ┌─────────┬─────────┬──────────┬────────┐
 │ OrderID ┆ Product ┆ Quantity ┆ Region │
 │ ---     ┆ ---     ┆ ---      ┆ ---    │
 │ i64     ┆ str     ┆ i64      ┆ str    │
 ╞═════════╪═════════╪══════════╪════════╡
 │ 4       ┆ C       ┆ 7        ┆ West   │
 │ 6   

In [78]:
sales.partition_by('Region', include_key=False)

[shape: (2, 3)
 ┌─────────┬─────────┬──────────┐
 │ OrderID ┆ Product ┆ Quantity │
 │ ---     ┆ ---     ┆ ---      │
 │ i64     ┆ str     ┆ i64      │
 ╞═════════╪═════════╪══════════╡
 │ 1       ┆ A       ┆ 10       │
 │ 3       ┆ A       ┆ 8        │
 └─────────┴─────────┴──────────┘,
 shape: (2, 3)
 ┌─────────┬─────────┬──────────┐
 │ OrderID ┆ Product ┆ Quantity │
 │ ---     ┆ ---     ┆ ---      │
 │ i64     ┆ str     ┆ i64      │
 ╞═════════╪═════════╪══════════╡
 │ 2       ┆ B       ┆ 5        │
 │ 5       ┆ B       ┆ 3        │
 └─────────┴─────────┴──────────┘,
 shape: (2, 3)
 ┌─────────┬─────────┬──────────┐
 │ OrderID ┆ Product ┆ Quantity │
 │ ---     ┆ ---     ┆ ---      │
 │ i64     ┆ str     ┆ i64      │
 ╞═════════╪═════════╪══════════╡
 │ 4       ┆ C       ┆ 7        │
 │ 6       ┆ A       ┆ 12       │
 └─────────┴─────────┴──────────┘]

In [80]:
sales_dict = sales.partition_by(['Region'], as_dict=True)
sales_dict

{('North',): shape: (2, 4)
 ┌─────────┬─────────┬──────────┬────────┐
 │ OrderID ┆ Product ┆ Quantity ┆ Region │
 │ ---     ┆ ---     ┆ ---      ┆ ---    │
 │ i64     ┆ str     ┆ i64      ┆ str    │
 ╞═════════╪═════════╪══════════╪════════╡
 │ 1       ┆ A       ┆ 10       ┆ North  │
 │ 3       ┆ A       ┆ 8        ┆ North  │
 └─────────┴─────────┴──────────┴────────┘,
 ('South',): shape: (2, 4)
 ┌─────────┬─────────┬──────────┬────────┐
 │ OrderID ┆ Product ┆ Quantity ┆ Region │
 │ ---     ┆ ---     ┆ ---      ┆ ---    │
 │ i64     ┆ str     ┆ i64      ┆ str    │
 ╞═════════╪═════════╪══════════╪════════╡
 │ 2       ┆ B       ┆ 5        ┆ South  │
 │ 5       ┆ B       ┆ 3        ┆ South  │
 └─────────┴─────────┴──────────┴────────┘,
 ('West',): shape: (2, 4)
 ┌─────────┬─────────┬──────────┬────────┐
 │ OrderID ┆ Product ┆ Quantity ┆ Region │
 │ ---     ┆ ---     ┆ ---      ┆ ---    │
 │ i64     ┆ str     ┆ i64      ┆ str    │
 ╞═════════╪═════════╪══════════╪════════╡
 │ 4       ┆ C 

In [82]:
# Coomo diccionario puedes acceder a las particiones por clave
sales_dict[('North',)]

OrderID,Product,Quantity,Region
i64,str,i64,str
1,"""A""",10,"""North"""
3,"""A""",8,"""North"""


## Takeaways

En este capítulo aprendimos a transformar la forma de tus datos usando Polars:

- **Formatos wide y long:** Comprendiste la diferencia entre datos anchos (wide) y largos (long).
- **De long a wide:** Usa `df.pivot()` para convertir datos largos en anchos.
- **De wide a long:** Usa `df.unpivot()` para convertir datos anchos en largos.
- **Transponer:** Usa `df.transpose()` para intercambiar filas y columnas.
- **Desanidar listas:** Usa `df.explode()` para descomponer listas anidadas en filas individuales.
- **Particionar:** Usa `df.partition_by()` para dividir un DataFrame en varios según una clave, incluso como diccionario.

Ahora puedes preparar tus datos para visualizarlos, tema que abordaremos en el siguiente capítulo.

In [ ]:
!uv pip install polars[gpu]

Using Python 3.12.11 environment at: C:\Users\sergi\Documents\polars\python-polars-the-definitive-guide\.venv
error: Failed to read metadata for: polars==1.32.3
  Caused by: failed to open file `C:\Users\sergi\Documents\polars\python-polars-the-definitive-guide\.venv\Lib\site-packages\polars-1.32.3.dist-info\METADATA`: El sistema no puede encontrar el archivo especificado. (os error 2)


: 

## GPU Engine Test

Polars puede usar GPU para acelerar operaciones en LazyFrames. Para usarlo, especifica `engine='gpu'` en `.collect()`.

**Requisitos:**
- GPU NVIDIA con CUDA
- `cudf-polars-cu12` instalado (incluido con `polars[gpu]`)

**Sintaxis en Polars 1.2+:**
```python
lf.collect(engine='gpu')  # Usa GPU si está disponible
lf.collect()              # CPU (por defecto)
```

Si no tienes GPU compatible, automáticamente usará CPU o lanzará un error según la operación.

In [4]:
# Probando GPU engine
import polars as pl

# Sintaxis correcta para Polars 1.2.1: engine='gpu' (string)
lf = pl.LazyFrame({
    "x": [1, 2, 3],
    "y": [10, 20, 30],
})

# Probar con GPU
try:
    result_gpu = lf.filter(pl.col("x") > 1).collect(engine='gpu')
    print("✅ GPU engine funcionando:")
    print(result_gpu)
except Exception as e:
    print(f"⚠️ GPU no disponible: {e}")
    # Fallback a CPU (por defecto)
    result_cpu = lf.filter(pl.col("x") > 1).collect()
    print("\n✅ Usando CPU engine:")
    print(result_cpu)

✅ GPU engine funcionando:
shape: (2, 2)
┌─────┬─────┐
│ x   ┆ y   │
│ --- ┆ --- │
│ i64 ┆ i64 │
╞═════╪═════╡
│ 2   ┆ 20  │
│ 3   ┆ 30  │
└─────┴─────┘
